# Smart Meal Food Catalog Image

This notebook creates a single PNG image showing the foods supported by the Smart Meal Keras model.

- If you do not provide a dataset folder, it creates labeled placeholder tiles.
- If you provide a dataset folder with one subfolder per class, it uses one sample image from each class.

Example dataset format:

```text
dataset/
  apple_fuji/
    image1.jpg
  samosa/
    image1.jpg
```

In [ ]:
from pathlib import Path
import json
import math
import textwrap

import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Circle
from PIL import Image, ImageOps

# Notebook location: backend/ml_models
BASE_DIR = Path.cwd()
CLASSES_PATH = BASE_DIR / 'class_names.json'
NUTRITION_PATH = BASE_DIR / 'nutrition_lookup.json'
OUTPUT_DIR = BASE_DIR / 'catalog'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Optional: set this to your dataset path if you have food images.
# Example: DATASET_DIR = Path(r'D:/datasets/smart_meal_foods')
DATASET_DIR = None

# Change this if you want fewer foods in the picture.
# Use None to show all 166 classes.
MAX_ITEMS = None

OUTPUT_IMAGE = OUTPUT_DIR / 'smart_meal_food_catalog.png'

In [ ]:
CATEGORY_RULES = [
    ('Fruit', ['apple', 'apricot', 'avocado', 'banana', 'grape', 'grapefruit', 'kiwi', 'lemon', 'lime', 'mango', 'melon', 'nectarine', 'orange', 'pear', 'plum', 'pomegranate', 'pomelo', 'strawberry', 'tangerine', 'watermelon']),
    ('Vegetable', ['beet', 'cabbage', 'carrot', 'corn', 'cucumber', 'daikon', 'garlic', 'onion', 'pepper', 'potato', 'pumpkin', 'raddish', 'tomato', 'zucchini']),
    ('Dessert', ['baklava', 'beignets', 'cake', 'cheesecake', 'chocolate', 'churros', 'creme_brulee', 'cup_cakes', 'donuts', 'frozen_yogurt', 'ice_cream', 'macarons', 'panna_cotta', 'pudding', 'red_velvet', 'shortcake', 'tiramisu', 'waffles']),
    ('Soup', ['bisque', 'chowder', 'soup', 'pho', 'ramen']),
    ('Salad', ['salad', 'guacamole', 'hummus']),
    ('Meat', ['beef', 'chicken', 'duck', 'filet', 'hamburger', 'hot_dog', 'pork', 'prime_rib', 'pulled_pork', 'ribs', 'steak']),
    ('Seafood', ['calamari', 'ceviche', 'crab', 'fish', 'lobster', 'mussels', 'oysters', 'salmon', 'sashimi', 'scallops', 'shrimp', 'sushi', 'tuna']),
    ('Rice/Noodles', ['bibimbap', 'fried_rice', 'gnocchi', 'pad_thai', 'paella', 'risotto']),
    ('Fast Food', ['fries', 'nachos', 'onion_rings', 'pizza', 'sandwich', 'tacos']),
    ('Breakfast', ['breakfast', 'eggs', 'french_toast', 'omelette', 'pancakes']),
    ('Snack/Appetizer', ['bruschetta', 'dumplings', 'edamame', 'falafel', 'gyoza', 'samosa', 'spring_rolls', 'takoyaki']),
    ('Pasta', ['lasagna', 'macaroni', 'ravioli', 'spaghetti']),
]

IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}

def display_name(food_id):
    return food_id.replace('_', ' ').title()

def infer_category(food_id):
    normalized = food_id.lower()
    for category, keywords in CATEGORY_RULES:
        if any(keyword in normalized for keyword in keywords):
            return category
    return 'Other'

def find_sample_image(dataset_dir, food_id):
    if dataset_dir is None:
        return None
    class_dir = Path(dataset_dir) / food_id
    if not class_dir.exists():
        return None
    for path in sorted(class_dir.iterdir()):
        if path.suffix.lower() in IMAGE_EXTENSIONS:
            return path
    return None

with open(CLASSES_PATH, 'r', encoding='utf-8') as f:
    class_names = json.load(f)

with open(NUTRITION_PATH, 'r', encoding='utf-8') as f:
    nutrition_db = json.load(f)

items = []
for food_id in class_names[:MAX_ITEMS]:
    nutrition = nutrition_db.get(food_id, {})
    items.append({
        'id': food_id,
        'name': display_name(food_id),
        'category': infer_category(food_id),
        'image_path': find_sample_image(DATASET_DIR, food_id),
        'calories': nutrition.get('calories', 0),
        'carbs': nutrition.get('carbs_g', 0),
        'sugar': nutrition.get('sugar_g', 0),
    })

len(items), items[:3]

In [ ]:
def draw_food_tile(ax, item):
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis('off')

    ax.add_patch(Rectangle((0.02, 0.02), 0.96, 0.96, facecolor='white', edgecolor='#d9dee7', linewidth=1.2))

    if item['image_path']:
        img = Image.open(item['image_path']).convert('RGB')
        img = ImageOps.fit(img, (260, 190), method=Image.Resampling.LANCZOS)
        ax.imshow(img, extent=(0.12, 0.88, 0.34, 0.88), aspect='auto')
    else:
        category_colors = {
            'Fruit': '#f97316', 'Vegetable': '#16a34a', 'Dessert': '#ec4899',
            'Soup': '#0ea5e9', 'Salad': '#22c55e', 'Meat': '#dc2626',
            'Seafood': '#2563eb', 'Rice/Noodles': '#ca8a04', 'Fast Food': '#ef4444',
            'Breakfast': '#7c3aed', 'Snack/Appetizer': '#0891b2', 'Pasta': '#d97706',
            'Other': '#64748b'
        }
        color = category_colors.get(item['category'], '#64748b')
        ax.add_patch(Circle((0.5, 0.63), 0.22, facecolor=color, edgecolor='none', alpha=0.95))
        initials = ''.join(word[0] for word in item['name'].split()[:2]).upper()
        ax.text(0.5, 0.63, initials, ha='center', va='center', fontsize=18, weight='bold', color='white')

    ax.text(0.5, 0.94, f"[{item['category']}]", ha='center', va='top', fontsize=7.5, weight='bold', color='#1d4ed8')
    wrapped_name = '\n'.join(textwrap.wrap(item['name'], width=18, max_lines=2))
    ax.text(0.5, 0.24, wrapped_name, ha='center', va='center', fontsize=8.5, weight='bold', color='#111827')
    nutrition_text = f"{item['calories']} kcal | {item['carbs']}g carbs | {item['sugar']}g sugar"
    ax.text(0.5, 0.105, nutrition_text, ha='center', va='center', fontsize=6.3, color='#4b5563')

columns = 8
rows = math.ceil(len(items) / columns)
fig_width = 18
fig_height = max(6, rows * 2.15)

fig, axes = plt.subplots(rows, columns, figsize=(fig_width, fig_height), dpi=180)
axes = axes.flatten() if hasattr(axes, 'flatten') else [axes]

for ax, item in zip(axes, items):
    draw_food_tile(ax, item)

for ax in axes[len(items):]:
    ax.axis('off')

title = f"Smart Meal Supported Foods ({len(class_names)} Keras Classes)"
subtitle = "Generated from class_names.json and nutrition_lookup.json"
if DATASET_DIR is None:
    subtitle += " | Add DATASET_DIR to use real food sample images"

fig.suptitle(title, fontsize=22, weight='bold', y=0.995)
fig.text(0.5, 0.982, subtitle, ha='center', va='top', fontsize=10, color='#475569')
plt.subplots_adjust(top=0.965, left=0.015, right=0.985, bottom=0.012, wspace=0.18, hspace=0.25)
fig.savefig(OUTPUT_IMAGE, bbox_inches='tight', facecolor='#f8fafc')
plt.close(fig)

print(f'Saved catalog image to: {OUTPUT_IMAGE}')